[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/autoencoder_vae/autoencoder_vae.ipynb)

In [ ]:
import os
import sys
import subprocess

# Default Colab GPU preference: A100 (fallback: L4)
if "google.colab" in sys.modules and not os.path.exists("benchmark.py"):
    if not os.path.exists("vision_transformers"):
        subprocess.run(
            ["git", "clone", "https://github.com/a-ramakrishnan/vision_transformers.git"],
            check=True,
        )
    os.chdir("vision_transformers")
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", "feat/larger-datasets-and-benchmarks"], check=True)
    os.chdir("autoencoder_vae")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
import json
import torch
from dataset import get_dataloaders
from models import ConvAutoencoder, ConvVAE
from train import train_autoencoder, train_vae, save_training_results
from visualize import plot_reconstructions, plot_vae_samples
from benchmark import benchmark_model, save_benchmark_results

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)
print(f"Using device: {device} ({gpu_name})")

DATASETS_TO_RUN = ["cifar10", "eurosat"]
BATCH_SIZE = 128
LATENT_DIM = 128
AE_EPOCHS = 15
VAE_EPOCHS = 20
LR = 1e-3

In [ ]:
all_training_results = {"runtime_gpu": gpu_name}
all_benchmark_results = {"runtime_gpu": gpu_name}

for ds_name in DATASETS_TO_RUN:
    print(f"\n==================== DATASET: {ds_name.upper()} ====================")
    train_loader, test_loader, cfg = get_dataloaders(dataset_name=ds_name, batch_size=BATCH_SIZE)
    in_ch, img_sz = cfg["in_channels"], cfg["image_size"]

    # 1. Convolutional Autoencoder
    ae_model = ConvAutoencoder(in_channels=in_ch, image_size=img_sz, latent_dim=LATENT_DIM).to(device)
    ae_model, ae_hist = train_autoencoder(
        ae_model, train_loader, epochs=AE_EPOCHS, lr=LR, device=device, checkpoint_path=f"ae_{ds_name}.pth"
    )
    plot_reconstructions(
        ae_model, test_loader, device=device, is_vae=False,
        title=f"{ds_name.upper()} ConvAutoencoder: Original (Top) vs Reconstructed (Bottom)",
        save_path=f"ae_{ds_name}_reconstructions.png",
    )
    ae_bench = benchmark_model(
        ae_model, test_loader, latent_dim=LATENT_DIM, device=device, is_vae=False, history=ae_hist, num_gen_samples=256
    )

    # 2. Convolutional VAE
    vae_model = ConvVAE(in_channels=in_ch, image_size=img_sz, latent_dim=LATENT_DIM).to(device)
    vae_model, vae_hist = train_vae(
        vae_model, train_loader, epochs=VAE_EPOCHS, lr=LR, kld_weight=1.0, device=device, checkpoint_path=f"vae_{ds_name}.pth"
    )
    plot_reconstructions(
        vae_model, test_loader, device=device, is_vae=True,
        title=f"{ds_name.upper()} ConvVAE: Original (Top) vs Reconstructed (Bottom)",
        save_path=f"vae_{ds_name}_reconstructions.png",
    )
    plot_vae_samples(
        vae_model, latent_dim=LATENT_DIM, num_samples=16, device=device,
        save_path=f"vae_{ds_name}_samples.png",
        title=f"{ds_name.upper()} ConvVAE: Generated Samples from Prior z ~ N(0, I)",
    )
    vae_bench = benchmark_model(
        vae_model, test_loader, latent_dim=LATENT_DIM, device=device, is_vae=True, history=vae_hist, num_gen_samples=256
    )

    all_training_results[ds_name] = {"autoencoder": ae_hist, "vae": vae_hist}
    all_benchmark_results[ds_name] = {"autoencoder": ae_bench, "vae": vae_bench}

    print(f"\n[{ds_name.upper()}] Autoencoder Benchmark:", json.dumps(ae_bench, indent=2))
    print(f"[{ds_name.upper()}] VAE Benchmark:", json.dumps(vae_bench, indent=2))

save_training_results(all_training_results, save_path="training_results.json")
save_benchmark_results(all_benchmark_results, save_path="benchmark_results.json")